In [76]:
import pandas as pd
import os
from abc import ABC, abstractmethod
from typing import Union, List
from pydantic.dataclasses import dataclass

In [77]:
os.listdir('../data/raw/2024/')[:5]

['INMET_CO_DF_A001_BRASILIA_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A042_BRAZLANDIA_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A045_AGUAS EMENDADAS_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A046_GAMA (PONTE ALTA)_01-01-2024_A_31-12-2024.CSV',
 'INMET_CO_DF_A047_PARANOA (COOPA-DF)_01-01-2024_A_31-12-2024.CSV']

In [78]:
csv_path = '../data/raw/2024/INMET_NE_BA_A401_SALVADOR_01-01-2024_A_31-12-2024.CSV'
df = pd.read_csv(csv_path, sep=';', encoding='latin-1', skiprows=lambda x: x in range(8))
 
df.head()

,Data,Hora UTC,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),RADIACAO GLOBAL (Kj/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",TEMPERATURA DO PONTO DE ORVALHO (°C),TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),"UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, DIREÇÃO HORARIA (gr) (° (gr))","VENTO, RAJADA MAXIMA (m/s)","VENTO, VELOCIDADE HORARIA (m/s)",Unnamed: 19
0,2024/01/01,0000 UTC,0,"1006,7","1006,7","1005,8",NaN,"26,8","22,8","26,8","26,6","22,8","22,6",79.0,78.0,79.0,66.0,"5,3","1,4",NaN
1,2024/01/01,0100 UTC,0,"1006,9","1006,9","1006,7",NaN,"26,7","22,6","26,8","26,5","22,8","22,4",79.0,77.0,78.0,62.0,"5,5","1,2",NaN
2,2024/01/01,0200 UTC,0,"1006,9","1007,1","1006,8",NaN,"26,5","22,9","26,7","26,4",23,"22,5",81.0,78.0,81.0,75.0,"4,6","1,2",NaN
3,2024/01/01,0300 UTC,0,"1006,5","1006,9","1006,5",NaN,"26,3","22,7","26,5","26,2","22,9","22,6",82.0,80.0,81.0,69.0,"4,6","1,1",NaN
4,2024/01/01,0400 UTC,0,"1006,5","1006,6","1006,5",NaN,26,"22,6","26,3","25,7","22,7","22,5",82.0,81.0,81.0,56.0,"4,2",",8",NaN


In [82]:
class IDataEngineering(ABC):

    @abstractmethod
    def read_inmet_data(self):
        pass

    @abstractmethod
    def default_read_csv(self):
        pass

    

In [ ]:
@dataclass
class DataEngInput:
    csv_paths: Union[str, List[str]]

class DataEngineering(IDataEngineering):
    def __init__(self, csv_paths:Union[str, List[str]]):

        self.csv_paths = DataEngInput(csv_paths=csv_paths)
        self.instance_of_csv = isinstance(self.csv_paths.csv_paths, str)

        self.read_inmet_data()


    def read_inmet_data(self):
        if self.instance_of_csv:
            self.raw_dataframes = self.default_read_csv(self.csv_paths.csv_paths)
        else:
            self.raw_dataframes = [self.default_read_csv(csv) for csv in self.csv_paths.csv_paths]

    
    def default_read_csv(self,_csv_path:str):
        return pd.read_csv(_csv_path, sep=';', encoding='latin-1', skiprows=lambda x: x in range(8))
    
    def cleaning_str_data_hours_columns(self, dataframe:pd.DataFrame):
        return dataframe.raw_dataframes['Data'] + dataframe.raw_dataframes['Hora UTC'].str.strip('UTC').str.strip(' ')

    def data_hours_str_to_datetime(self):
        if self.instance_of_csv:
            df_ = self.raw_dataframes
            df_['DATA_HORA'] = self.cleaning_str_data_hours_columns(self.raw_dataframes)
            df_.drop(['Data', 'Hora UTC'], axis=1)
        
        else:
            _aux_dataframe_list = []
            for dataframe in self.raw_dataframes:
                dataframe['DATA_HORA'] = self.cleaning_str_data_hours_columns(dataframe)
                dataframe.drop(['Data', 'Hora UTC'], axis=1)
                _aux_dataframe_list.append(dataframe)
            self.raw_dataframes = _aux_dataframe_list
            


In [112]:
data_test = DataEngineering(csv_paths='../data/raw/2024/INMET_NE_BA_A401_SALVADOR_01-01-2024_A_31-12-2024.CSV')

In [113]:
data_test.data_hours_str_to_datetime()

AttributeError: 'DataFrame' object has no attribute 'raw_dataframes'

In [ ]:
pd.to_datetime(data_test.raw_dataframes['Data'] + data_test.raw_dataframes['Hora UTC'].str.strip('UTC').str.strip(' '),
               format='%Y/%m/%d%H%M', 
               errors='coerce')

0      2024-01-01 00:00:00
1      2024-01-01 01:00:00
2      2024-01-01 02:00:00
3      2024-01-01 03:00:00
4      2024-01-01 04:00:00
               ...        
8779   2024-12-31 19:00:00
8780   2024-12-31 20:00:00
8781   2024-12-31 21:00:00
8782   2024-12-31 22:00:00
8783   2024-12-31 23:00:00
Length: 8784, dtype: datetime64[us]